In [16]:
import os

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import scanpy as sc
import torch

import T_perturb
from T_perturb.Perturb.datamodule import PerturberDataModule
from T_perturb.Perturb.trainer import PerturberGenerationTrainer, PerturberInferenceTrainer

from T_perturb.src.utils import label_encoder
from T_perturb.tests.test_cellgen_training import dummy_dataset
from T_perturb.tests.test_countdecoder_training import dummy_cell_gene_matrix

In [17]:
import sys
print("\nKernel directory path:")
print(os.path.dirname(sys.executable))


Kernel directory path:
/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/bin


In [18]:
if os.getcwd().split('/')[-1] != 'healthy_imm_expr':
    # set working directory to root of repository
    os.chdir('/lustre/scratch126/cellgen/team361/kl11/t_generative/')

In [19]:
tgt_vocab_size = 101  # +1 for padding token
num_samples = 100
num_genes = 100
max_seq_length = 50
n_total_tps = 2
num_samples = 100
batch_size = 4
pred_tps = [1, 2]
context_tps = [1, 2]
d_model = 12

genes_to_perturb = [70]
perturbation_token = 1


In [20]:
src_counts = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
)
src_dataset = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
)
tgt_counts_dict = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
    total_time_steps=n_total_tps,
)
tgt_datasets = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
    total_time_steps=n_total_tps,
)

In [21]:
type(src_dataset)

datasets.arrow_dataset.Dataset

In [22]:
conditions = None
condition_keys = None
conditions_combined = None

In [23]:
if condition_keys is None:
    condition_keys = 'tmp_batch'
    # create a mock vector if there are no batch effect
    tmp_series = pd.DataFrame(
        {
            condition_keys: np.ones(num_samples),
        }
    )


In [24]:
if isinstance(condition_keys, str):
    condition_keys_ = [condition_keys]
else:
    condition_keys_ = condition_keys

if conditions is None:
    if condition_keys is not None:
        conditions_ = {}
        for cond in condition_keys_:
            conditions_[cond] = tmp_series[cond].unique().tolist()
    else:
        conditions_ = {}
else:
    conditions_ = conditions

if conditions_combined is None:
    if len(condition_keys_) > 1:
        tmp_series['conditions_combined'] = tmp_series[condition_keys].apply(
            lambda x: '_'.join(x), axis=1
        )
    else:
        tmp_series['conditions_combined'] = tmp_series[condition_keys]
    conditions_combined_ = tmp_series['conditions_combined'].unique().tolist()
else:
    conditions_combined_ = conditions_combined

condition_encodings = {
    cond: {k: v for k, v in zip(conditions_[cond], range(len(conditions_[cond])))}
    for cond in conditions_.keys()
}
conditions_combined_encodings = {
    k: v for k, v in zip(conditions_combined_, range(len(conditions_combined_)))
}

In [25]:
tgt_adata_tmp = sc.AnnData(X=tgt_counts_dict['tgt_h5ad_t1'].squeeze(), obs=tmp_series)

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [26]:
if (condition_encodings is not None) and (condition_keys_ is not None):
    conditions = [
        label_encoder(
            tgt_adata_tmp,
            encoder=condition_encodings[condition_keys_[i]],
            condition_key=condition_keys_[i],
        )
        for i in range(len(condition_encodings))
    ]
    conditions = torch.tensor(conditions, dtype=torch.long).T
    conditions_combined = label_encoder(
        tgt_adata_tmp,
        encoder=conditions_combined_encodings,
        condition_key='conditions_combined',
    )
    conditions_combined = torch.tensor(conditions_combined, dtype=torch.long)


In [31]:
import importlib
import T_perturb.src.utils
importlib.reload(T_perturb.Perturb.trainer)
importlib.reload(T_perturb.Perturb.datamodule)
importlib.reload(T_perturb.src.utils)
from T_perturb.Perturb.trainer import PerturberInferenceTrainer
from T_perturb.Perturb.datamodule import PerturberDataModule

trainer_params = {
    'tgt_vocab_size': tgt_vocab_size,
    'd_model': d_model,
    'num_heads': 4,
    'num_layers': 1,
    'd_ff': 8,
    'max_seq_length': max_seq_length + 10,
    'dropout': 0.0,
    'pred_tps': pred_tps,
    'context_tps': context_tps,
    'n_total_tps': n_total_tps,
    'precision': 'high',
    'mask_scheduler': 'pow',
    'output_dir': './T_perturb/T_perturb/tests/res',
    'encoder': 'Transformer_encoder',
    'var_list': None,
    'genes_to_perturb': genes_to_perturb,
    'perturbation_token': perturbation_token,
    'context_mode': True,
    'perturbation_mode': ['src', 'tgt'],
}
# decoder_module = PerturberInferenceTrainer(
#     weight_decay=0.0,
#     end_lr=1e-3,
#     return_embeddings=True,
#     **trainer_params
#     )
decoder_module = PerturberGenerationTrainer(
    ckpt_masking_path='T_perturb/T_perturb/tests/checkpoints/test_masking_checkpoint-epoch=00.ckpt',
    ckpt_count_path='T_perturb/T_perturb/tests/checkpoints/test_counts_checkpoint-epoch=00.ckpt',
    loss_mode='zinb',
    lr=1e-3,
    weight_decay=0.0,
    conditions=conditions_,
    conditions_combined=conditions_combined_,
    temperature=1.5,
    iterations=19,
    sequence_length=max_seq_length - 10,
    n_samples=3,
    seed=42,
    n_genes=num_genes,
    **trainer_params
)

celltype_to_perturb = 'A'
data_module = PerturberDataModule(
    src_counts=src_counts,
    tgt_counts_dict=tgt_counts_dict,
    src_dataset=src_dataset,
    tgt_datasets=tgt_datasets,
    batch_size=batch_size,
    num_workers=1,
    pred_tps=pred_tps,
    context_tps=context_tps,
    n_total_tps=n_total_tps,
    split=False,
    max_len=max_seq_length,
    condition_keys=condition_keys_,
    condition_encodings=condition_encodings,
    conditions=conditions,
    conditions_combined=conditions_combined,
    celltype_to_perturb=celltype_to_perturb,
    filter_mode='exclude'
)
data_module.setup()
test_loader = data_module.test_dataloader()

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/lustre/scratch126/cellgen/team361/kl11/t_generative/T_perturb/T_perturb/Model/trainer.py:550: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_

Using CPU for training
test kwargs {'ckpt_masking_path': 'T_perturb/T_perturb/tests/checkpoints/test_masking_checkpoint-epoch=00.ckpt', 'ckpt_count_path': 'T_perturb/T_perturb/tests/checkpoints/test_counts_checkpoint-epoch=00.ckpt', 'loss_mode': 'zinb', 'lr': 0.001, 'weight_decay': 0.0, 'conditions': {'tmp_batch': [1.0]}, 'conditions_combined': [1.0], 'temperature': 1.5, 'iterations': 19, 'sequence_length': 40, 'n_samples': 3, 'seed': 42, 'n_genes': 100, 'tgt_vocab_size': 101, 'd_model': 12, 'num_heads': 4, 'num_layers': 1, 'd_ff': 8, 'max_seq_length': 60, 'dropout': 0.0, 'pred_tps': [1, 2], 'context_tps': [1, 2], 'n_total_tps': 2, 'precision': 'high', 'mask_scheduler': 'pow', 'output_dir': './T_perturb/T_perturb/tests/res', 'encoder': 'Transformer_encoder', 'var_list': None, 'context_mode': True}
Start datamodule
src_dataset_filtered Dataset({
    features: ['input_ids', 'cell_type', 'length', 'cell_pairing_index'],
    num_rows: 67
})
[100, 100, 100, 100, 100, 100, 100, 100, 100, 100

/lustre/scratch126/cellgen/team361/kl11/t_generative/T_perturb/T_perturb/Model/trainer.py:570: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_cou

In [32]:
trainer = pl.Trainer(
    limit_test_batches=1,  # Limit to a single batch for quick testing
    logger=False,
)
trainer.test(decoder_module, data_module)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_test_batches=1)` was configured so 1 batch will be used.


src_dataset_filtered Dataset({
    features: ['input_ids', 'cell_type', 'length', 'cell_pairing_index'],
    num_rows: 67
})
[100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
src_len 67
tgt_len 66


/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Testing DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]1
perturbating tgt
2
perturbating tgt
perturbating src
Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 201.09it/s]


[{}]